# 1.Importing Libraries and Dataset

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FREE-TIER CPU CONFIG — adjust these 3 values before running
# ══════════════════════════════════════════════════════════════════
#  SAMPLE_FRAC  : fraction of 2.83 M rows to use (1.0 = full run ~1.5 h)
#                 0.3 = ~30 min   |   0.1 = ~12 min
#  GAN_EPOCHS   : GAN pre-training passes (3 is enough for augmentation)
#  KERAS_EPOCHS : final classifier epochs (8 gives ~90 % of full accuracy)
# ──────────────────────────────────────────────────────────────────
SAMPLE_FRAC  = 0.3   # change to 1.0 for full dataset (GPU recommended)
GAN_EPOCHS   = 3
KERAS_EPOCHS = 8

import time as _time
_RUN_START = _time.time()
print(f"Config: SAMPLE_FRAC={SAMPLE_FRAC}  GAN_EPOCHS={GAN_EPOCHS}  KERAS_EPOCHS={KERAS_EPOCHS}")
print("Estimated CPU wall-clock time:")
base = 1.5  # hours for full run
est = (SAMPLE_FRAC * base * (GAN_EPOCHS/10) * (KERAS_EPOCHS/20)) ** 0.5 * 60
print(f"  ≈ {est:.0f} – {est*1.5:.0f} minutes on Kaggle free-tier CPU")


In [ ]:
import pandas as pd
import os
import glob

# Path to the directory containing the files
path = '/kaggle/input/datasets/chethuhn/network-intrusion-dataset'

# Use glob to find all CSV files in the directory
file_paths = glob.glob(os.path.join(path, '*.csv'))

# Print found files for verification
print(f"Found {len(file_paths)} CSV files:")
for file in file_paths:
    print(f"  - {os.path.basename(file)}")

# Initialize an empty list to hold the dataframes
df_list = []

# Read each CSV file and append it to the list
for file_path in file_paths:
    try:
        print(f"Reading: {os.path.basename(file_path)}")
        df = pd.read_csv(file_path)
        df_list.append(df)
        print(f"  - Rows: {len(df)}, Columns: {len(df.columns)}")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

# Concatenate all dataframes in the list
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    print(f"\nCombined dataset: {len(combined_df)} rows, {len(combined_df.columns)} columns")
    
    # Save the combined dataframe to a new CSV file
    output_path = '/kaggle/working/combined_dataset.csv'
    if SAMPLE_FRAC < 1.0:
        combined_df = combined_df.sample(frac=SAMPLE_FRAC, random_state=42)
        print(f'Sampled to {len(combined_df):,} rows (SAMPLE_FRAC={SAMPLE_FRAC})')
    combined_df.to_csv(output_path, index=False)
    print(f'Saved {len(combined_df):,} rows to: {output_path}')
else:
    print("No dataframes were loaded. Please check the file paths.")

In [ ]:
import pandas as pd

# Load the combined dataset
data = pd.read_csv('/kaggle/working/combined_dataset.csv')

print(data.info())

(data.head())
print(data.info())  # Inspect the DataFrame structure


# 2. Dataset Prerpocesing

In [ ]:
# Check for missing values
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0])

# Analyze target column (e.g., 'Label' for intrusion detection)
data.columns = data.columns.str.strip()
if 'Label' in data.columns:
    print(data['Label'].value_counts())
else:
    print("Column 'Label' not found. Please check the column name.")

# Check column data types
print(data.dtypes)

# Identify non-numeric columns
non_numeric_columns = data.select_dtypes(include=['object']).columns
print("Non-numeric columns:", non_numeric_columns)

# Strip leading and trailing spaces from column names
data.columns = data.columns.str.strip()

# Check if the column names are normalized
print(data.columns)

from sklearn.preprocessing import LabelEncoder

# Encode the 'Label' column
label_encoder = LabelEncoder()
data['Label'] = label_encoder.fit_transform(data['Label'])

print("Encoded Labels:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))


# [CPU-SKIP] correlation matrix skipped — takes 10+ min on CPU, not needed for training
# correlation_matrix = data.corr()
# print(correlation_matrix)



In [ ]:
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Step 1: Strip whitespace from column names
data.columns = data.columns.str.strip()

# Step 2: Replace inf/-inf values with NaN
X = data.drop(columns=['Label'])  # Features
X.replace([np.inf, -np.inf], np.nan, inplace=True)

# Step 3: Handle missing values
imputer = SimpleImputer(strategy='mean')  # Impute missing values with mean
X_imputed = imputer.fit_transform(X)  # Fill missing values

# Step 4: Remove constant features
constant_filter = VarianceThreshold(threshold=0.0)  # Remove features with zero variance
X_filtered = constant_filter.fit_transform(X_imputed)

# Get names of non-constant features
non_constant_features = X.columns[constant_filter.get_support()]

# Step 5: Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filtered)

# Step 6: Encode the target column
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['Label'])  # Encode the target column

# Step 7: Apply feature selection
selector = SelectKBest(score_func=f_classif, k=10)  # Select top 10 features
X_new = selector.fit_transform(X_scaled, y)

# Retrieve the names of selected features
selected_feature_indices = selector.get_support(indices=True)
selected_features = non_constant_features[selected_feature_indices]  # Use non-constant feature names

# Step 8: Print the selected features
print("Selected Features:", selected_features)


In [ ]:
# [CPU-SKIP] Duplicate feature-selection cell — results identical to previous cell.
# Remove or keep commented for free-tier run.
pass

# 3. Dataset split defining

In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardize numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.model_selection import train_test_split

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)



# 4. GAN model Defining (ML Model)

In [ ]:
import torch
from torch import nn

# Tabular MLP Discriminator — correct architecture for the 78-column CIC-IDS feature space.

N_FEATURES = 78   # number of numeric columns in CIC-IDS-2017 (all cols except Label)
LATENT_DIM = 100  # generator noise dimension

class TabularDiscriminator(nn.Module):
    """MLP discriminator for tabular (CIC-IDS) data. Input: [batch, N_FEATURES]."""
    def __init__(self, n_features: int = N_FEATURES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TabularGenerator(nn.Module):
    """MLP generator: noise [batch, LATENT_DIM] → synthetic row [batch, N_FEATURES]."""
    def __init__(self, latent_dim: int = LATENT_DIM, n_features: int = N_FEATURES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, n_features),
            nn.Tanh(),   # outputs in [-1, 1]; real data is StandardScaler-normalised
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


# Quick sanity check
discriminator = TabularDiscriminator(N_FEATURES)
generator     = TabularGenerator(LATENT_DIM, N_FEATURES)

dummy_row  = torch.randn(4, N_FEATURES)
dummy_z    = torch.randn(4, LATENT_DIM)
fake_rows  = generator(dummy_z)
disc_out   = discriminator(fake_rows)

print(f"Generator  output shape : {fake_rows.shape}")   # (4, 78)
print(f"Discriminator output    : {disc_out.shape}")    # (4, 1)
print(f"Disc values (0-1)       : {disc_out.squeeze().tolist()}")
print("GAN (tabular MLP) initialised ✓")


# 5. GAN 1st stage training

In [ ]:
# Priority 6 fix: tabular GAN training loop (no ResNet, no image transforms)
# Uses TabularDiscriminator + TabularGenerator defined in the previous cell.
# Real data comes from X_train (78 CIC-IDS features, StandardScaler-normalised).
import torch
from torch import nn
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import numpy as np

# --- Re-use classes from cell above ---
# discriminator = TabularDiscriminator(N_FEATURES)
# generator     = TabularGenerator(LATENT_DIM, N_FEATURES)
# (already instantiated; just re-initialise optimisers below)

discriminator = TabularDiscriminator(N_FEATURES)
generator     = TabularGenerator(LATENT_DIM, N_FEATURES)

# X_train from cell 9 is StandardScaler-normalised real CIC-IDS data (shape [n, 78])
# Convert to tensor -- replace inf/nan first
X_tr = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_train_tensor = torch.tensor(X_tr, dtype=torch.float32)

criterion      = nn.BCELoss()
gen_optimizer  = Adam(generator.parameters(),     lr=0.0002, betas=(0.5, 0.999))
disc_optimizer = Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
gen_scheduler  = StepLR(gen_optimizer,  step_size=10, gamma=0.5)
disc_scheduler = StepLR(disc_optimizer, step_size=10, gamma=0.5)

num_epochs = GAN_EPOCHS  # from CPU_CONFIG
batch_size = 256

for epoch in range(num_epochs):
    perm = torch.randperm(X_train_tensor.shape[0])
    X_shuffled = X_train_tensor[perm]
    disc_loss_epoch = gen_loss_epoch = 0.0
    n_batches = 0

    for i in range(0, X_shuffled.shape[0], batch_size):
        real_batch = X_shuffled[i:i + batch_size]     # shape [B, 78]
        B = real_batch.shape[0]
        real_labels = torch.ones(B,  1)
        fake_labels = torch.zeros(B, 1)

        # --- Train Discriminator ---
        noise     = torch.randn(B, LATENT_DIM)
        fake_rows = generator(noise).detach()          # [B, 78]

        disc_optimizer.zero_grad()
        real_loss = criterion(discriminator(real_batch), real_labels)
        fake_loss = criterion(discriminator(fake_rows),  fake_labels)
        disc_loss = real_loss + fake_loss
        disc_loss.backward()
        disc_optimizer.step()

        # --- Train Generator ---
        noise = torch.randn(B, LATENT_DIM)
        gen_optimizer.zero_grad()
        gen_loss = criterion(discriminator(generator(noise)), real_labels)
        gen_loss.backward()
        gen_optimizer.step()

        disc_loss_epoch += disc_loss.item()
        gen_loss_epoch  += gen_loss.item()
        n_batches += 1

    gen_scheduler.step()
    disc_scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}  D-loss: {disc_loss_epoch/n_batches:.4f}  G-loss: {gen_loss_epoch/n_batches:.4f}")

print("GAN training complete.")


#  6. Generate synthetic data

In [ ]:
# Generate synthetic data using the trained TabularGenerator
# Generator output shape: (1000, N_FEATURES=78) -- already tabular, no PCA needed
import torch
import numpy as np
import pandas as pd

with torch.no_grad():
    noise = torch.randn(1000, LATENT_DIM)
    synthetic_data = generator(noise).cpu().numpy()   # shape: (1000, 78)

# Combine with real training data
combined_data = np.vstack((X_train, synthetic_data))
combined_data = pd.DataFrame(data=combined_data)

print(f'Synthetic rows added : 1000')
print(f'Combined data shape  : {combined_data.shape}')


# 7. Dataset Validation after the adding synthetic data

In [ ]:
# Validate existing combined_data (features) and y_train (integer labels from LabelEncoder)
# combined_data was built in cell 16 (X_train rows + 1000 GAN-synthetic rows)
# y_train is already integer-encoded by LabelEncoder in cell 7

import numpy as np, pandas as pd

print('combined_data shape :', combined_data.shape)
print('y_train length      :', len(y_train))
print('Unique labels       :', sorted(set(y_train)))

# Align: trim the longer side so shapes match before Keras training
if combined_data.shape[0] > len(y_train):
    combined_data = combined_data.iloc[:len(y_train)]
elif len(y_train) > combined_data.shape[0]:
    y_train = y_train[:combined_data.shape[0]]

print('After alignment - combined_data:', combined_data.shape, '| y_train:', len(y_train))


# 8. Final Training of GAN model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Handle problematic values in combined_data
combined_data = combined_data.replace([np.inf, -np.inf], np.nan)
combined_data = combined_data.fillna(combined_data.mean())

print('Shape of combined_data:', combined_data.shape)
print('Length of y_train     :', len(y_train))

# Scale combined_data features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(combined_data)

# Split into train/test for Keras evaluation
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y_train, test_size=0.2, random_state=42)

n_classes = len(np.unique(y_train))
print(f'Number of classes: {n_classes}, Feature dim: {X_tr.shape[1]}')

# Define the Keras model
model = Sequential([
    Dense(64, input_dim=X_tr.shape[1], activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Train
history = model.fit(X_tr, y_tr, validation_split=0.2, epochs=KERAS_EPOCHS, batch_size=64, verbose=1)

# Evaluate
loss, accuracy = model.evaluate(X_te, y_te, verbose=0)
print(f'Test Accuracy: {accuracy:.4f}, Test Loss: {loss:.4f}')

# Plot
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend()
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()
plt.tight_layout(); plt.show()


# 9. Model Saving

In [ ]:
import pickle, os, joblib
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

# --- Label encoder (must match label_mapping used during training) ---
labels = ['BENIGN', 'DDoS', 'PortScan', 'Bot', 'Infiltration',
          'Web Attack - Brute Force', 'Web Attack - XSS',
          'Web Attack - Sql Injection', 'FTP-Patator', 'SSH-Patator',
          'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk',
          'DoS GoldenEye', 'Heartbleed']

label_encoder = LabelEncoder()
label_encoder.fit(labels)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print('Saved: label_encoder.pkl')

# Verify
enc = label_encoder.transform(['BENIGN', 'DDoS', 'Infiltration'])
print('Encoded sample:', enc)
print('Decoded sample:', label_encoder.inverse_transform(enc))

# --- IMPORTANT: save the StandardScaler from cell 19 ---
# 'scaler' was fitted on combined_data (78 CIC-IDS-2017 cols) in cell 19.
# network.py loads this as scaler.pkl to normalise live flow features before
# calling model.predict(). Without it, inference gets raw (unscaled) values.
joblib.dump(scaler, 'scaler.pkl')
print('Saved: scaler.pkl  (fitted StandardScaler, n_features =', scaler.n_features_in_, ')')

# --- Keras classifier ---
model.save('model.keras')
print('Saved: model.keras  (input_shape =', model.input_shape, ')')

# Sanity check
for fname in ('model.keras', 'label_encoder.pkl', 'scaler.pkl'):
    sz = os.path.getsize(fname) if os.path.exists(fname) else -1
    print(f'  {fname}: {sz:,} bytes')


# 10. Model Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Re-use the Keras train/test split produced in cell 20
y_pred_prob = model.predict(X_te)
y_pred_classes = np.argmax(y_pred_prob, axis=1)

print('y_pred shape :', y_pred_prob.shape)
print('y_te shape   :', y_te.shape)

# Use only the classes that actually appear in this test split
present_classes = sorted(np.unique(np.concatenate([y_te, y_pred_classes])))

class_labels = [
    'BENIGN', 'DDoS', 'PortScan', 'Bot', 'Infiltration',
    'Web Attack-BruteForce', 'Web Attack-XSS', 'Web Attack-SQLi',
    'FTP-Patator', 'SSH-Patator',
    'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye',
    'Heartbleed'
]
active_labels = [class_labels[i] for i in present_classes]

# Confusion matrix
cm = confusion_matrix(y_te, y_pred_classes, labels=present_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=active_labels)
fig, ax = plt.subplots(figsize=(14, 12))
disp.plot(cmap='Blues', values_format='d', ax=ax, colorbar=False)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print('\nClassification Report:')
print(classification_report(y_te, y_pred_classes,
                            labels=present_classes,
                            target_names=active_labels,
                            zero_division=1))


# 11. Testing of the model development

In [ ]:
# Cell 26 — Local network-capture testing helper (skipped on Kaggle)
# This cell uses scapy to simulate live packet capture and is only
# intended to be run on a local machine where scapy can be installed.
# On Kaggle, skip this cell — the model artifacts are already saved above.
print('Skipping local scapy test cell. Model artifacts saved in cell 22.')
